# 02 · Image understanding (Gemini)

How photos are turned into symptoms. Needs `GEMINI_API_KEY` in `.env` and
`pip install google-genai pillow python-dotenv`.

1. The plain Gemini call (basic version).
2. The app's safer version (`app/vision.py`): validates the image, restricts the answer to our symptom vocabulary.
3. Feeding the detected symptoms into the disease model.

In [ ]:
import os, sys, json
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

IMAGE_PATH = ROOT / "patient_image.jpg"      # <- change to your own test photo
assert os.getenv("GEMINI_API_KEY"), "Set GEMINI_API_KEY in .env"
assert IMAGE_PATH.exists(), f"Put a test image at {IMAGE_PATH}"

## 1 · Plain Gemini call

In [ ]:
from google import genai
from PIL import Image

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

def analyze_image(image_path, question="Describe this image."):
    image = Image.open(image_path)
    response = client.models.generate_content(model="gemini-2.5-flash", contents=[question, image])
    return response.text

print(analyze_image(IMAGE_PATH, "Analyze this image and describe the visible symptoms."))

## 2 · The app's analyzer
Differences from the plain call: size/format/pixel validation, downscaling to 1024 px, JSON output,
symptoms limited to the model's vocabulary (anything else is discarded), and an instruction to
describe visible signs only, never to diagnose.

In [ ]:
from app.preprocessing import SymptomPreprocessor
from app.vision import ImageAnalyzer, load_image

vocab = json.loads((ROOT / "artifacts" / "vocabulary.json").read_text())
analyzer = ImageAnalyzer(os.getenv("GEMINI_API_KEY"), "gemini-2.5-flash", SymptomPreprocessor(vocab))
findings = analyzer.analyze([load_image(IMAGE_PATH.read_bytes())])
print("symptoms   :", findings.symptoms)
print("description:", findings.description)

## 3 · From image findings to ranked conditions

In [ ]:
from app.model import SymptomModel

model = SymptomModel()
vector = model.preprocessor.vectorize(findings.symptoms)
for c in model.predict(vector, top_k=5):
    print(f"{c.probability:6.1%}  {c.name}")